# 🚀 Visual Regression AI Training — Kaggle (续训)

**运行前设置：**
- 右上角 → **Session options** → Accelerator → **GPU T4 x2**
- 左侧 **+Add Data** 上传两个文件：
  1. `FYP_VISUAL_colab.zip`（最新项目代码）
  2. `visual_ai.ckpt.pt`（从 Lightning AI 下载的存档）
- 运行完毕后点右上角 **Save Version → Save & Run All (Commit)**

## ✅ Step 0 — 确认 GPU 配置

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  VRAM: {p.total_memory/1024**3:.1f} GB')

if torch.cuda.device_count() == 0:
    raise RuntimeError('❌ 没有 GPU！请在 Session options 里选 GPU T4 x2 再重新 Commit')
elif torch.cuda.device_count() == 1:
    print('⚠️  只有 1 张 GPU，建议选 T4 x2 获得双卡加速')
else:
    print(f'✅ {torch.cuda.device_count()} 张 GPU — DataParallel 会自动启用')

## 📦 Step 1 — 上传并解压项目

先点左侧 **+Add Data → Upload** 把 `FYP_VISUAL_colab.zip` 上传为 Dataset，
然后运行下面的 Cell

In [ ]:
import os, zipfile, shutil, sys
from pathlib import Path

WORK_DIR = Path('/kaggle/working/project')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

# Option A: zip file uploaded
possible_zips = list(Path('/kaggle/input').rglob('FYP_VISUAL_colab.zip'))
if possible_zips:
    print(f'找到 zip: {possible_zips[0]}，解压中...')
    with zipfile.ZipFile(str(possible_zips[0]), 'r') as zf:
        zf.extractall(WORK_DIR)
else:
    # Option B: project files already extracted in dataset
    possible_roots = list(Path('/kaggle/input').rglob('visual_regression/__init__.py'))
    if possible_roots:
        ROOT_SRC = possible_roots[0].parent.parent
        print(f'找到项目文件: {ROOT_SRC}，复制中...')
        shutil.copytree(str(ROOT_SRC), str(WORK_DIR), dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            '找不到项目文件！\n'
            '请上传 FYP_VISUAL_colab.zip 或整个项目文件夹到 Dataset'
        )

hits = list(WORK_DIR.rglob('visual_regression/__init__.py'))
if not hits:
    raise RuntimeError('找不到 visual_regression/ 目录')
ROOT = hits[0].parent.parent
os.chdir(ROOT)
print(f'✅ 工作目录: {ROOT}')

In [ ]:
解压完成后，按顺序继续跑下面的 cell。

## 📦 Step 2 — 安装依赖

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
!pip install -q scikit-image datasets huggingface_hub

from visual_regression.ai_training import train_model
print('✅ 所有模块 import 正常')

## 🔁 Step 2.5 — 复制 checkpoint（续训用）

In [ ]:
import shutil
from pathlib import Path

ckpt_candidates = list(Path('/kaggle/input').rglob('visual_ai.ckpt.pt'))
ckpt_dest = Path('/kaggle/working/project/.visual-regression/models/visual_ai.ckpt.pt')
ckpt_dest.parent.mkdir(parents=True, exist_ok=True)

if ckpt_candidates:
    shutil.copy(str(ckpt_candidates[0]), str(ckpt_dest))
    print(f'✅ Checkpoint 已复制: {ckpt_candidates[0]}')
else:
    print('ℹ️  没有找到 visual_ai.ckpt.pt，将从 epoch 1 开始训练')

## 🌐 Step 3 — 下载训练数据集

In [ ]:
import subprocess, json, sys
from pathlib import Path

print('⏳ 开始下载数据集（预计 20-40 分钟）...')
subprocess.run([sys.executable, 'download_datasets.py'], cwd=str(ROOT))

MANIFEST = ROOT / '.visual-regression/datasets/public-ui-manifest.json'
if MANIFEST.exists():
    data = json.loads(MANIFEST.read_text())
    print(f'✅ 数据集就绪: {data.get("total_images", "?")} 张图片')
else:
    print('⚠️  Manifest 未生成，将使用内置合成图像训练')
    MANIFEST = None

## 🧠 Step 4 — 训练 AI

In [ ]:
import time, torch, multiprocessing
from visual_regression.config import WorkspacePaths
from visual_regression.ai_training import train_model, StreamingSyntheticDataset

# Kaggle: 2 workers (3 copies x 4.5GB = 13.5GB < 30GB RAM)
multiprocessing.cpu_count = lambda: 4
StreamingSyntheticDataset._MAX_SRC_PX = 800

paths = WorkspacePaths(ROOT)
paths.ensure()

EPOCHS = 10
SAMPLES_PER_IMAGE = 24
BATCH_SIZE = 64
LEARNING_RATE = 1e-4

MODEL_OUT = ROOT / '.visual-regression/models/visual_ai.pt'
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

ckpt = ROOT / '.visual-regression/models/visual_ai.ckpt.pt'
print('=' * 55)
print('  Visual Regression AI Training')
print(f'  Device  : {"CUDA x" + str(torch.cuda.device_count()) if torch.cuda.is_available() else "CPU"}')
print(f'  Epochs  : {EPOCHS}  |  Max px: {StreamingSyntheticDataset._MAX_SRC_PX}')
print(f'  Workers : 2  |  Checkpoint: {"resume" if ckpt.exists() else "fresh"}')
print('=' * 55)

t0 = time.time()
metadata = train_model(
    paths=paths,
    model_path=MODEL_OUT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    samples_per_image=SAMPLES_PER_IMAGE,
    pixel_threshold=20,
    min_region_area=120,
    pretrained_backbone=True,
    dataset_manifest_path=MANIFEST,
    max_public_images=5000,
)
elapsed = time.time() - t0
print(f'\n Training done! {elapsed/3600:.1f}h')
print(f'   Accuracy: {metadata["accuracy"]:.2%}')

## Step 5 — 保存模型

In [ ]:
import shutil
from pathlib import Path

OUTPUT = Path('/kaggle/working')
shutil.copy2(str(MODEL_OUT), str(OUTPUT / 'visual_ai.pt'))

meta_src = MODEL_OUT.with_suffix('.json')
if meta_src.exists():
    shutil.copy2(str(meta_src), str(OUTPUT / 'visual_ai.json'))

ckpt_src = MODEL_OUT.with_suffix('.ckpt.pt')
if ckpt_src.exists():
    shutil.copy2(str(ckpt_src), str(OUTPUT / 'visual_ai.ckpt.pt'))
    print('✅ ckpt 已保存（下次续训用）')

pt_size = (OUTPUT / 'visual_ai.pt').stat().st_size / 1024**2
print(f'✅ 模型: visual_ai.pt ({pt_size:.1f} MB)')
print('   → Output 面板 Download 下载')